# Reproducing Churchland et al. (2010) from dandiset 000138

**Paper.** Churchland MM, Cunningham JP, Kaufman MT, Ryu SI, Shenoy KV (2010). Cortical
preparatory activity: representation of movement or first cog in a dynamical machine? *Neuron*
68:387-400. [doi:10.1016/j.neuron.2010.09.015](https://doi.org/10.1016/j.neuron.2010.09.015). The
claims file (`claims/000138/10.1016-j.neuron.2010.09.015.yaml`) was written from the NIH author
manuscript in PubMed Central (PMC2991102) with its supplement; the main-text figure panels were
not available to its reader.

**Dandiset.** [DANDI:000138](https://dandiarchive.org/dandiset/000138/0.220113.0407), version
0.220113.0407 (MC_Maze_Large, Neural Latents Benchmark). The paper (2010) predates the dandiset
and cites no version. This is the only published version, and the draft holds the same two files
(same blobs, checked below), so the most recent published version is pinned.

**What it holds.** One session of monkey J (2009-10-06) on the maze task: 500 successful trials
over 27 conditions, 162 sorted units from two Utah arrays, hand, cursor and eye position, and hand
velocity. The paper's J-array dataset had 108 conditions, 146 units and a mean 2166 trials per
neuron; the dandiset is a 500-trial subset of a session, and nothing in it or the paper says
whether this session is the one the paper used. The single-electrode datasets (monkeys A, B and
J) and all EMG are not in it.

**Readings.** The claims file names several open method details and the paper leaves a few
more open. Each is run where it could change an outcome; a main reading, the one best supported by
the paper, decides an outcome when the readings disagree, and the outcome is then marked as
resting on an assumption. Seeds are fixed per claim.

| Claim | Tested | Outcome |
|---|---|---|
| fig4abc-tuning-correlation-single-electrode | no | blocked, data missing |
| fig4d-tuning-correlation-j-array | yes | partially reproduced (assumption) |
| fig4e-correlation-time-course | yes | not reproduced |
| results-ap-location-no-effect | J-array only | reproduced (assumption) |
| supp-preparatory-dimensionality | J-array only | reproduced (assumption) |
| fig6-peri-movement-space-best | J-array only | reproduced |
| fig6-each-dimensionality | J-array only | reproduced (assumption) |
| fig6-dimensionality-trend | J-array only | reproduced (assumption) |
| fig6-pd-adequate-all-spaces | J-array only | reproduced (assumption) |
| fig6e-summary-rank | no | blocked, data missing |
| fig7a-shuffled-space-poor | J-array only | reproduced (assumption) |
| fig7b-window-start-onset | J-array only | reproduced (assumption) |
| fig7c-anterior-posterior | J-array only | not reproduced (assumption) |
| fig7d-pd-consistency | J-array only | reproduced |
| fig8-oscillator-model | simulation | reproduced |

In [ ]:
import itertools
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import prepspace as ps

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
data = ps.load_train()
trials, units, spikes, hand = data["trials"], data["units"], data["spikes"], data["hand"]
print(f"loaded dandiset {ps.DANDISET_ID} version {ps.DANDISET_VERSION}, {ps.TRAIN_PATH}, "
      f"in {time.time() - started:.0f} s")

## Data check

The archive's listing of the pinned version and of the draft, and the file's metadata.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/000138"
versions = requests.get(f"{api}/versions/").json()["results"]
print("Versions:", [(v["version"], v["asset_count"], v["size"]) for v in versions])
for v in ["0.220113.0407", "draft"]:
    assets = requests.get(f"{api}/versions/{v}/assets/").json()["results"]
    print(v, [(a["path"], a["size"], a["blob"]) for a in assets])
meta = requests.get(f"{api}/versions/{ps.DANDISET_VERSION}/").json()
print("\nRelated resources:", [(r["identifier"], r["relation"]) for r in meta["relatedResource"]])
print("Description:", meta["description"])
print("\nFile metadata:", data["meta"])

Both versions hold the same two blobs, so the published version is the draft's content. The
training file holds trials, units and behaviour; the test file holds 100 unlabelled trial
segments for the benchmark (checked below).

In [ ]:
print("Trials:", len(trials), "; success:", trials.success.value_counts().to_dict(),
      "; split:", trials.split.value_counts().to_dict())
print("Conditions (trial_type x trial_version):", trials.condition.nunique(),
      "; maze_id values:", trials.maze_id.nunique())
print(trials.groupby(["trial_type", "trial_version"]).size().unstack().to_string())
check = trials.groupby("condition").agg(maze_ids=("maze_id", "nunique"),
                                        barriers=("num_barriers", lambda x: sorted(set(x))),
                                        targets=("num_targets", lambda x: sorted(set(x))),
                                        active=("active_target", lambda x: sorted(set(x))))
print("\nPer condition, distinct maze_id, number of barriers, targets, active target index:")
print(check.to_string())
print("\ndelay field minus (go cue - target on) in ms, max abs:",
      float(np.abs(trials.delay - (trials.go_cue_time - trials.target_on_time) * 1000).max()))
print("rt field minus (move onset - go cue) in ms, max abs:",
      float(np.abs(trials.rt - (trials.move_onset_time - trials.go_cue_time) * 1000).max()))
print("Delay (ms) range:", trials.delay.min(), "to", trials.delay.max(),
      "; trials with delay > 400 ms:", int((trials.delay > 400).sum()))
print("Trials with delay of 0 ms or less:", int((trials.delay <= 0).sum()))
print("RT (ms) range:", trials.rt.min(), "to", trials.rt.max())
print("Target on after trial start (s):", round((trials.target_on_time - trials.start_time).min(), 3),
      "to", round((trials.target_on_time - trials.start_time).max(), 3))
print("Trial stop after movement onset (s):", round((trials.stop_time - trials.move_onset_time).min(), 3),
      "to", round((trials.stop_time - trials.move_onset_time).max(), 3))

The `delay` and `rt` fields agree with the event times to rounding. Eight trials have a delay
of 0 ms or less (down to -2 ms); the 400 ms rule excludes them. Each condition is one maze
(`maze_id`) in one version: version 0 has one target and no barriers, version 1 one target
with barriers, version 2 three targets with barriers, as the field descriptions say, and as the
Supplemental Experimental Procedures describe (straight reaches, curved reaches, and curved
reaches with distractors). The paper's J-array dataset had 108 conditions and a mean 2166
trials per neuron; this file has 27 conditions and 500 trials (about 18 per condition).

In [ ]:
el = data["electrodes"]
print("Electrodes:", len(el), "; by group:", el.group_name.value_counts().to_dict(),
      "; x, y, z all NaN:", bool(el[["x", "y", "z"]].isna().all().all()))
print("Units:", len(units), "; heldout:", units.heldout.value_counts().to_dict())
print("Location of the electrode each unit references:", units.electrode_location.value_counts().to_dict())
print("Unit id = array * 1000 + channel * 10 + sort: id prefix counts",
      units.id_array.value_counts().sort_index().to_dict())
print("Channel in the id equals the referenced electrode index + 1 for every unit:",
      bool((units.id_channel == units.electrode + 1).all()))
print("Heldout units by id prefix:", units.groupby("id_array").heldout.sum().to_dict())
n_spk = np.array([len(s) for s in spikes])
print("Spikes per unit:", n_spk.min(), "to", n_spk.max(), "; total", n_spk.sum())
print("Spike times sorted in every unit:", all(np.all(np.diff(s) >= 0) for s in spikes))
starts, stops = trials.start_time.values, trials.stop_time.values
allsp = np.concatenate(spikes)
k = np.searchsorted(starts, allsp, side="right") - 1
print("Spikes outside the trial intervals:", int(np.sum((k < 0) | (allsp > stops[np.clip(k, 0, None)]))))
dur = (stops - starts).sum()
rates = n_spk / dur
print(f"Mean rate per unit over the trial intervals (spikes/s): {rates.min():.2f} to {rates.max():.2f}, "
      f"median {np.median(rates):.2f}")

The units' electrode references all point into the first 96 rows of the electrodes table,
the PMd group, and every electrode coordinate is NaN. The unit ids carry a prefix of 1 (83
units) or 2 (79 units) and the channel, and the referenced electrode is the channel minus
one for both prefixes, so the 79 units of prefix 2 reference PMd electrodes by the same channel
numbers as prefix 1. The heldout units are split 20 and 20 between the prefixes. The data
description says the units come from an M1 and a PMd array. The prefix is consistent with the
array a unit was recorded on, but which prefix is which array is not stated anywhere in the
files; that the references are wrong for one prefix is a finding.

In [ ]:
hp, hv, ht = hand["hand_pos"], hand["hand_vel"], hand["t"]
for name in ["hand_pos", "hand_vel", "cursor_pos", "eye_pos"]:
    x = hand[name]
    print(f"{name}: stored unit {hand[name + '_unit']!r}, conversion {hand[name + '_conversion']}, "
          f"description {hand[name + '_description']!r}; x {x[:, 0].min():.1f} to {x[:, 0].max():.1f}, "
          f"y {x[:, 1].min():.1f} to {x[:, 1].max():.1f}")
dt = np.diff(ht)
print("Hand samples:", len(ht), "; interval within trials", np.median(dt) * 1000, "ms; samples outside trials:",
      int(np.sum(ht > stops[np.searchsorted(starts, ht, side='right') - 1] + 1e-9)))
# velocity against the derivative of position, within trials
within = np.r_[False, np.isclose(dt, 0.001)]
dpos = np.r_[np.zeros((1, 2)), np.diff(hp, axis=0)] * 1000
fast = within & (np.hypot(hv[:, 0], hv[:, 1]) > 100)
for d, ax in enumerate("xy"):
    r = np.corrcoef(dpos[fast, d], hv[fast, d])[0, 1]
    slope = np.polyfit(dpos[fast, d], hv[fast, d], 1)[0]
    print(f"hand_vel {ax} against the derivative of hand_pos {ax} (samples above 100 mm/s): r = {r:.3f}, slope {slope:.3f}")
# reach endpoint against the active target
end_err = []
for _, tr in trials.iterrows():
    i = np.searchsorted(ht, tr.move_onset_time + 0.75)
    target = np.asarray(tr.target_pos)[int(tr.active_target)]
    end_err.append(np.hypot(*(hp[i] - target)))
end_err = np.array(end_err)
print(f"Hand 750 ms after movement onset to the active target (mm): median {np.median(end_err):.1f}, "
      f"95th percentile {np.percentile(end_err, 95):.1f}, max {end_err.max():.1f}")

Hand position is stored in millimetres with a conversion of 0.001 to metres, and velocity in
mm/s; velocity agrees with the derivative of position (slope near 1). Hand samples exist only
inside trials, at 1 ms; the paper tracked the hand at 60 Hz, so the 1 kHz series is
resampled. Hand positions 750 ms after movement onset lie near the active target, which ties
the hand, the target positions and the movement onset times together.

Clock check. If spikes share the trial table's clock, the reach target can be decoded from
spike counts after target onset and not before it; if hand data share it, hand speed rises at
the table's movement onset. Target (one of 9 mazes, `trial_type`) is decoded with a
leave-one-trial-out nearest-centroid classifier on square-root spike counts (a standard
choice), in 200 ms windows. The same is done with the windows shifted 1 s earlier, which
moves every window before target onset.

In [ ]:
def decode(t0_s, window):
    i0 = np.round((t0_s + window[0] / 1000) * 1000).astype(int)
    counts = np.zeros((len(trials), len(spikes)))
    for u, s in enumerate(spikes):
        ms = np.floor(s * 1000).astype(int)
        cs = np.r_[0, np.cumsum(np.bincount(ms, minlength=int(stops.max() * 1000) + 3000))]
        counts[:, u] = cs[i0 + window[1] - window[0]] - cs[i0]
    X = np.sqrt(counts)
    y = trials.trial_type.values
    labels = np.unique(y)
    sums = np.stack([X[y == l].sum(0) for l in labels])
    ns = np.array([(y == l).sum() for l in labels])
    correct = 0
    for i in range(len(y)):
        li = np.searchsorted(labels, y[i])
        cent = sums.copy()
        cent[li] -= X[i]
        nn = ns.copy()
        nn[li] -= 1
        cent = cent / nn[:, None]
        correct += labels[np.argmin(((cent - X[i]) ** 2).sum(1))] == y[i]
    return correct / len(y)

clock = []
for name, ev, win in [("target onset", trials.target_on_time.values, (-250, -50)),
                      ("target onset", trials.target_on_time.values, (100, 300)),
                      ("go cue", trials.go_cue_time.values, (-200, 0)),
                      ("movement onset", trials.move_onset_time.values, (0, 200))]:
    clock.append({"event": name, "window_ms": win, "accuracy": decode(ev, win),
                  "accuracy_window_1s_earlier": decode(ev - 1.0, win) if name == "target onset" else np.nan})
print("Chance for 9 mazes about", round(1 / 9, 3))
print(pd.DataFrame(clock).round(3).to_string(index=False))
speed_on = []
for tm in trials.move_onset_time.values:
    i = np.searchsorted(ht, tm - 0.0005)
    speed_on.append(np.hypot(*hv[i - 100:i + 101].T))
speed_on = np.array(speed_on)
print("Median hand speed (mm/s) at -100, -50, 0, +50, +100 ms from the table's movement onset:",
      np.round(np.median(speed_on, 0)[[0, 50, 100, 150, 200]], 1))

The test file: what it holds, and whether its segments duplicate training trials. A run of
five successive inter-spike intervals of a unit is looked up among the same unit's training
intervals (rounded to 0.1 ms).

In [ ]:
test = ps.load_test_summary()
print("Test file trial columns:", test["trial_columns"], "; trials:", test["n_trials"],
      "; units:", len(test["unit_ids"]),
      "; unit ids all among the training file's held-in units:",
      set(test["unit_ids"]) == set(units.id[~units.heldout]))
print("Segment durations (s):", np.unique(np.round(test["stops"] - test["starts"], 3)))
hits, tried = 0, 0
for uid, st in zip(test["unit_ids"][:20], test["spikes"][:20]):
    train_isi = np.round(np.diff(spikes[int(np.nonzero(units.id.values == uid)[0][0])]) * 1e4).astype(int)
    train_keys = {tuple(train_isi[i:i + 5]) for i in range(len(train_isi) - 4)}
    isi = np.round(np.diff(st) * 1e4).astype(int)
    for i in range(0, len(isi) - 4, 5):
        tried += 1
        hits += tuple(isi[i:i + 5]) in train_keys
print(f"Runs of 5 test intervals found in the training intervals of the same unit: {hits} of {tried}")

The test segments carry no condition, so the analysis uses the training file only (its
`train` and `val` trials, 500 in all).

Trial selection under each reading (Methods: delay over 400 ms; Supplemental Experimental
Procedures: velocity profiles correlated with a proto-trial at r > 0.9, "typically ~90% of all
trials"). Movement onset reading (b) is the speed threshold crossing.

In [ ]:
onsets = {"a": ps.onset_times(data, "a"), "b": ps.onset_times(data, "b")}
d_on = (onsets["b"] - onsets["a"]) * 1000
print(f"Onset (b) minus onset (a), ms: median {np.median(d_on):.0f}, 5th to 95th percentile "
      f"{np.percentile(d_on, 5):.0f} to {np.percentile(d_on, 95):.0f}, range {d_on.min():.0f} to {d_on.max():.0f}")
t0 = time.time()
R = ps.trial_rates(data, onsets)
print(f"Smoothed rates computed in {time.time() - t0:.0f} s")
ctxs = {}
rows = []
for proto, combine, onset in itertools.product(ps.READINGS["proto"], ps.READINGS["combine"], ps.READINGS["onset"]):
    ctx = ps.prepare(data, R, onsets, proto, combine, onset)
    ctxs[(proto, combine, onset)] = ctx
    per = np.bincount(trials.cond_idx[ctx["mask"]], minlength=27)
    row = {"proto": proto, "combine": combine, "onset": onset, "kept": int(ctx["mask"].sum()),
           "of_delay_over_400": int((trials.delay > 400).sum()),
           "fraction": ctx["mask"].sum() / (trials.delay > 400).sum(),
           "min_per_condition": per.min(), "max_per_condition": per.max()}
    for tun in ps.READINGS["tuning"]:
        tp = ps.tuning(ctx["prep"], tun)
        row[f"included_tuning_{tun}"] = int((tp > ps.ARRAY_MIN_TUNING).sum())
    rows.append(row)
selection = pd.DataFrame(rows)
print(selection.round(3).to_string(index=False))
per_cond = pd.DataFrame({k: np.bincount(trials.cond_idx[c["mask"]], minlength=27) for k, c in ctxs.items()}).T
per_cond.index.names = ["proto", "combine", "onset"]
print("\nTrials kept per condition (columns: condition index):")
print(per_cond.to_string())

Requiring the horizontal and the vertical profile each to correlate at r > 0.9 (`each`)
keeps about 80% of trials and leaves as few as 2 or 3 trials in a condition: in reaches that
are nearly vertical or horizontal, one velocity component is small and its correlation with
the proto-trial is mostly noise. Correlating the concatenated profile (`concat`) keeps about
95%. The paper reports "typically ~90%".

Under the range reading of tuning (a) about 90 to 100 of the 162 units pass the array
criterion of more than 5 spikes/s; under the SD reading (b) only 6 or 7 do. Ranges and units
of the measures used:

In [ ]:
ctx = ctxs[("a", "concat", "a")]
tp_a, tp_b = ps.tuning(ctx["prep"], "a"), ps.tuning(ctx["prep"], "b")
print(f"Preparatory condition means (spikes/s): {ctx['prep'].min():.2f} to {ctx['prep'].max():.2f}")
print(f"Preparatory tuning, range (spikes/s): {tp_a.min():.2f} to {tp_a.max():.2f}, median {np.median(tp_a):.2f}")
print(f"Preparatory tuning, SD (spikes/s): {tp_b.min():.2f} to {tp_b.max():.2f}, median {np.median(tp_b):.2f}")
peri = ps.epoch_mean(ctx["move"], ps.PERI_WINDOW)
print(f"Peri-movement condition means (spikes/s): {peri.min():.2f} to {peri.max():.2f}")
inc_a = tp_a > ps.ARRAY_MIN_TUNING
print("Included units (range reading) by id prefix:", pd.Series(units.id_array[inc_a]).value_counts().sort_index().to_dict())
print("Median preparatory tuning (range) by id prefix:",
      pd.Series(tp_a).groupby(units.id_array.values).median().round(2).to_dict())
print("Median peri-movement tuning (range) by id prefix:",
      pd.Series(peri.max(1) - peri.min(1)).groupby(units.id_array.values).median().round(2).to_dict())
print(f"Hand position, condition means (mm): x {ctx['pos'][..., 0].min():.0f} to {ctx['pos'][..., 0].max():.0f}, "
      f"y {ctx['pos'][..., 1].min():.0f} to {ctx['pos'][..., 1].max():.0f}; "
      f"peak speed {np.hypot(ctx['vel'][..., 0], ctx['vel'][..., 1]).max():.0f} mm/s")

The paper found preparatory activity more prevalent at anterior sites (Results, Comparison of
anterior versus posterior sites). The prefix with more preparatory tuning would then be the
PMd array; this is printed above and used only as a consistency check of the assumption made
below that prefix 1 is PMd, the group the electrode references name.

## Readings

The claims file and the paper leave several method details open. Each is run. The first
value listed is the main reading, chosen as best supported by the paper (the reasons are in
the attempt record):

- `proto`: proto-trial as the condition's mean (a) or elementwise median (b) velocity profile.
- `combine`: the r > 0.9 test on the concatenated horizontal and vertical profile, or on each.
- `onset`: movement onset from the trial table (a) or a speed threshold (b).
- `tuning`: preparatory tuning as the range (a) or SD (b) of condition means.
- `t_units`: all units (a) or only included units (b) in the peri-movement T-matrix.
- `fold`: PCA space computed once (a) or recomputed without the left-out condition (b).
- `norm`: performance per neuron, then averaged (a), or pooled (b).
- `kin_end`: kinematic window ends at +800 ms (a) or at the end of the reach (b).
- `task_end`: reach end by speed threshold (a) or at +800 ms (b), for task spaces.
- `split`: halfpoint and segments by path length (a) or time (b).
- `kmax`: dimensionalities 1 to 14 (a) or 1 to 12 (b).

In [ ]:
print(ps.MAIN)

## fig4abc-tuning-correlation-single-electrode

> In the three single-electrode datasets (monkeys A, B, J) the correlation across conditions
> between a neuron's preparatory and peri-movement firing rates is positive on average but
> small.

Not tested: the dandiset holds only array data from monkey J. The single-electrode recordings
of monkeys A, B and J are not in it.

## fig4d-tuning-correlation-j-array

> In the J-array dataset the mean correlation between preparatory and peri-movement tuning is
> positive but small (mean r 0.13, p = 0.03; mean Spearman r 0.11).

Per neuron, Pearson and Spearman correlation over the 27 conditions between preparatory
(go -200 to +100 ms) and peri-movement (movement onset -100 to +350 ms) rates, for units with
more than 5 spikes/s of preparatory tuning and peri-movement tuning at least 1.5 times that.
Tests of the mean against zero: (a) one-sample t-test on r, (b) the same on Fisher z, (c)
Wilcoxon signed-rank.

In [ ]:
def fig4_selection(ctx, tun, onset):
    prep = ctx["prep"]
    peri = ps.epoch_mean(ctx["move"], ps.PERI_WINDOW)
    tp, tq = ps.tuning(prep, tun), ps.tuning(peri, tun)
    inc = (tp > ps.ARRAY_MIN_TUNING) & (tq >= ps.PERI_RATIO * tp)
    return inc, prep, peri

rows = []
fig4 = {}
for (proto, combine, onset), ctx in ctxs.items():
    for tun in ps.READINGS["tuning"]:
        inc, prep, peri = fig4_selection(ctx, tun, onset)
        row = {"proto": proto, "combine": combine, "onset": onset, "tuning": tun,
               "n_units": int(inc.sum())}
        if inc.sum() >= 3:
            r, rs = ps.corr_pearson_spearman(prep[inc], peri[inc])
            fig4[(proto, combine, onset, tun)] = (inc, r, rs)
            row.update({"mean_r": r.mean(), "mean_spearman": rs.mean(),
                        "p_t": stats.ttest_1samp(r, 0).pvalue,
                        "p_t_fisher": stats.ttest_1samp(np.arctanh(np.clip(r, -0.999999, 0.999999)), 0).pvalue,
                        "p_wilcoxon": stats.wilcoxon(r).pvalue})
            row["r_within_tol"] = abs(row["mean_r"] - 0.13) <= 0.013
            row["spearman_within_tol"] = abs(row["mean_spearman"] - 0.11) <= 0.011
            row["positive_p05"] = row["mean_r"] > 0 and min(row["p_t"], row["p_t_fisher"], row["p_wilcoxon"]) < 0.05
        rows.append(row)
fig4d = pd.DataFrame(rows)
print("Expected: mean r 0.13 +/- 0.013, mean Spearman 0.11 +/- 0.011, mean r > 0 at p < 0.05")
print(fig4d.round(4).to_string(index=False))
main4 = fig4d[(fig4d.proto == "a") & (fig4d["combine"] == "concat") & (fig4d.onset == "a") & (fig4d.tuning == "a")].iloc[0]
print("\nMain reading:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in main4.to_dict().items()})
ok = fig4d.dropna(subset=["mean_r"]).astype({"r_within_tol": bool, "spearman_within_tol": bool, "positive_p05": bool})
print("Readings with a result:", len(ok), "; both means within tolerance:",
      int((ok.r_within_tol & ok.spearman_within_tol).sum()), "; mean r positive at p < 0.05:", int(ok.positive_p05.sum()))
print("Range of mean r over readings with a result:", round(ok.mean_r.min(), 3), "to", round(ok.mean_r.max(), 3),
      "; mean Spearman:", round(ok.mean_spearman.min(), 3), "to", round(ok.mean_spearman.max(), 3))

In [ ]:
inc, r, rs = fig4[("a", "concat", "a", "a")]
print("Main reading, per-unit Pearson r: range", round(r.min(), 3), "to", round(r.max(), 3),
      "; by id prefix:", pd.Series(r).groupby(units.id_array.values[inc]).agg(["count", "mean"]).round(3).to_dict())
# how the result moves with the peri-movement criterion
for ratio in [1.0, 1.25, 1.5, 2.0]:
    ctx = ctxs[("a", "concat", "a")]
    prep = ctx["prep"]; peri = ps.epoch_mean(ctx["move"], ps.PERI_WINDOW)
    tp, tq = ps.tuning(prep, "a"), ps.tuning(peri, "a")
    sel = (tp > 5) & (tq >= ratio * tp)
    rr, rrs = ps.corr_pearson_spearman(prep[sel], peri[sel])
    print(f"peri/prep tuning ratio >= {ratio}: {sel.sum()} units, mean r {rr.mean():.3f}, "
          f"mean Spearman {rrs.mean():.3f}, t-test p {stats.ttest_1samp(rr, 0).pvalue:.4f}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(r, bins=np.linspace(-1, 1, 21), color="0.5")
ax.axvline(r.mean(), color="r")
ax.set_xlabel("Pearson r, preparatory vs peri-movement")
ax.set_ylabel("units")
ax.set_title("Main reading (all values inside the axes)")
plt.show()

**Result: partially reproduced, under the main reading of trial selection and movement onset (an
assumption).** Under the main reading 20 units pass the criteria. Mean r is 0.195 and mean
Spearman r 0.212, both above the expected 0.13 +/- 0.013 and 0.11 +/- 0.011, and the mean r is
above zero (t-test p = 0.023, on Fisher z p = 0.023, Wilcoxon p = 0.024), which by the criterion
is a partial reproduction. Over the eight readings with a result (all with tuning as a range),
mean r spans 0.105 to 0.195 and mean Spearman r 0.147 to 0.212; mean r is within tolerance in
five readings, mean Spearman r in none, and the mean r is above zero at p < 0.05 in two; in the
other six it is positive, with the smallest p of the three tests from 0.074 to 0.169. Under the SD reading of tuning no unit passes. With a peri-movement criterion of 1.0 in
place of 1.5, 52 units give mean r 0.137 (p = 0.0041). The dandiset has 27 of the paper's 108
conditions.

## fig4e-correlation-time-course

> The correlation of preparatory tuning with peri-movement tuning measured at single time
> points starts to fall about 200 ms before movement onset and is near zero by movement onset.

Same units as Figure 4D. The rate at each time point is taken from the movement-locked
average (the movement-locked part of the paper's spliced trace); the go-locked curve is shown
alongside.

In [ ]:
tt = np.arange(*ps.MOVE_TRACE)
rows = []
curves = {}
for key, (inc, r, rs) in fig4.items():
    ctx = ctxs[key[:3]]
    prep = ctx["prep"][inc]
    trace = ctx["move"][inc]
    pc = prep - prep.mean(1, keepdims=True)
    tc = trace - trace.mean(1, keepdims=True)
    curve = np.nanmean((pc[:, :, None] * tc).sum(1) / np.sqrt((pc ** 2).sum(1)[:, None] * (tc ** 2).sum(1)), 0)
    curves[key] = curve
    at = {ms: curve[ms - ps.MOVE_TRACE[0]] for ms in [-400, -300, -250, -200, -100, 0]}
    rows.append({"proto": key[0], "combine": key[1], "onset": key[2], "tuning": key[3], "n_units": int(inc.sum()),
                 **{f"r_at_{ms}": v for ms, v in at.items()},
                 "near_zero_at_0": abs(at[0]) < 0.1, "below_half_of_-250": at[0] < 0.5 * at[-250],
                 "no_early_decline": at[-300] >= 0.75 * at[-400]})
fig4e = pd.DataFrame(rows)
fig4e["criterion_met"] = fig4e["near_zero_at_0"] & fig4e["below_half_of_-250"] & fig4e["no_early_decline"]
print(fig4e.round(3).to_string(index=False))
print("Readings meeting the criterion:", int(fig4e.criterion_met.sum()), "of", len(fig4e))
main_curve = curves[("a", "concat", "a", "a")]
print("Main reading, mean correlation every 50 ms from -600 to +400 ms:")
print(pd.Series(main_curve[::50], index=tt[::50]).round(3).to_string())

In [ ]:
inc = fig4[("a", "concat", "a", "a")][0]
ctx = ctxs[("a", "concat", "a")]
gt = np.arange(-200, 500)
gtrace = ps.condition_means(R["go_trace"], trials, ctx["mask"])[inc]
prep = ctx["prep"][inc]
pc = prep - prep.mean(1, keepdims=True)
gc = gtrace - gtrace.mean(1, keepdims=True)
go_curve = np.nanmean((pc[:, :, None] * gc).sum(1) / np.sqrt((pc ** 2).sum(1)[:, None] * (gc ** 2).sum(1)), 0)
print("Go-locked mean correlation every 50 ms from -200 to +450 ms after the go cue:")
print(pd.Series(go_curve[::50], index=gt[::50]).round(3).to_string())
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
axs[0].plot(gt, go_curve, "k"); axs[0].set_xlabel("ms from go cue")
axs[1].plot(tt, main_curve, "k"); axs[1].set_xlabel("ms from movement onset")
for a in axs:
    a.axhline(0, color="0.7"); a.set_ylim(-0.4, 1.05)
axs[0].set_ylabel("mean r with preparatory rates")
plt.show()
print("Curve values outside the axes:", int(np.sum((main_curve < -0.4) | (main_curve > 1.05)) + np.sum((go_curve < -0.4) | (go_curve > 1.05))))

**Result: not reproduced.** The mean correlation is 0.773 at -400 ms and 0.764 at -300 ms,
0.713 at -250 ms and 0.572 at -200 ms, so the decline begins about 250 ms before movement onset,
as the paper describes. It is 0.218 at movement onset, not near zero by the criterion (below
0.1), and first falls below 0.1 between +150 and +200 ms (0.110 and 0.064). All eight readings
give 0.129 to 0.218 at onset, and all meet the other two parts of the criterion. Every time point the
criterion uses (-400 ms to onset) lies inside the movement-locked trace, so the splice points
of the paper's trace do not enter.

## results-ap-location-no-effect

> The tuning correlation of Figure 4 does not depend on anterior-posterior recording location
> in any dataset (r = -0.05, -0.03, -0.07 and 0.20; p = 0.82, 0.85, 0.52 and 0.27).

Only the J-array dataset can be tested. Electrode coordinates are NaN, so reading (a), the AP
coordinate of the electrode, cannot be run. Reading (b), array membership, needs the array of
each unit; it is taken from the unit id prefix, assuming prefix 1 is the PMd array (anterior,
coded 1) and prefix 2 the M1 array (posterior, coded 0). The per-unit Pearson r of Figure 4D
is regressed on that code; the slope's p-value is the paper's (r and p of the regression), and
p of 0.05 or more is "no effect" (assumed threshold, as the claims file says).

In [ ]:
rows = []
for key, (inc, r, rs) in fig4.items():
    ap = (units.id_array.values[inc] == 1).astype(float)
    if len(set(ap)) < 2:
        continue
    lr = stats.linregress(ap, r)
    rows.append({"proto": key[0], "combine": key[1], "onset": key[2], "tuning": key[3],
                 "n_anterior": int(ap.sum()), "n_posterior": int((1 - ap).sum()),
                 "mean_r_anterior": r[ap == 1].mean(), "mean_r_posterior": r[ap == 0].mean(),
                 "regression_r": lr.rvalue, "p": lr.pvalue, "no_effect_p_ge_05": lr.pvalue >= 0.05})
ap_tab = pd.DataFrame(rows)
print(ap_tab.round(3).to_string(index=False))
print("Readings with no effect (p >= 0.05):", int(ap_tab.no_effect_p_ge_05.sum()), "of", len(ap_tab))

**Result: reproduced for the J-array dataset, on the assumption that the unit id prefix gives the
array, prefix 1 being PMd.** Regressing the per-unit correlation on array gives r = 0.047,
p = 0.844 under the main reading (12 anterior and 8 posterior units; mean r 0.209 and 0.176),
and p from 0.432 to 0.844 over the eight readings, so no reading finds an effect. The paper's
J-array values are r = 0.20, p = 0.27. The single-electrode datasets are not in the dandiset.

## supp-preparatory-dimensionality

> Depending on the dataset, 7 to 10 dimensions capture 90% of the variance of preparatory
> activity across conditions.

PCA of the 27 x n matrix of preparatory condition means, rates not normalized; components
for 90% of variance. Units entering: (a) all, (b) included.

In [ ]:
def n_for_90(M):
    Mc = M - M.mean(0)
    s = np.linalg.svd(Mc, compute_uv=False) ** 2
    return int(np.searchsorted(np.cumsum(s) / s.sum(), 0.9) + 1), np.cumsum(s) / s.sum()

rows = []
for (proto, combine, onset), ctx in ctxs.items():
    M = ctx["prep"].T
    n_all, cum_all = n_for_90(M)
    row = {"proto": proto, "combine": combine, "onset": onset, "units_all": M.shape[1], "k90_all": n_all}
    for tun in ps.READINGS["tuning"]:
        inc = ps.tuning(ctx["prep"], tun) > ps.ARRAY_MIN_TUNING
        row[f"units_included_{tun}"] = int(inc.sum())
        row[f"k90_included_{tun}"] = n_for_90(M[:, inc])[0]
    rows.append(row)
dim = pd.DataFrame(rows)
print("Expected: 7 to 10, widened to 6.3 to 11")
print(dim.to_string(index=False))
_, cum = n_for_90(ctxs[("a", "concat", "a")]["prep"].T)
print("Main reading, all units, cumulative variance of the first 15 PCs:", np.round(cum[:15], 3))
# the same on a split of trials, to see how much of the 90% is trial noise
rng = np.random.default_rng(4)
ctx = ctxs[("a", "concat", "a")]
half = ctx["mask"] & (rng.random(len(trials)) < 0.5)
other = ctx["mask"] & ~half
A = ps.condition_means(R["prep"], trials, half).T
B = ps.condition_means(R["prep"], trials, other).T
print("Split halves of trials (seed 4): k90 =", n_for_90(A)[0], "and", n_for_90(B)[0])
print("Trial counts per condition in each half:", np.bincount(trials.cond_idx[half], minlength=27).min(), "to",
      np.bincount(trials.cond_idx[half], minlength=27).max())

**Result: reproduced under the main reading (an assumption).** With all 162 units, 11 components
capture 90% of the variance, inside 6.3 to 11; with the 92 included units, 10. Under the `each`
trial selection the counts are 12 and 11, so with all units the count is outside the range; under
the SD reading of tuning, with 6 or 7 units, it is 4. On half of the trials the count rises to 14
and 13, so trial noise adds dimensions at these trial counts. The count of 11 sits at the upper end of
the range, so the outcome is fragile.

## PD performance in each space (Figure 6, J-array)

One grid of PD performance serves the Figure 6 claims and Figure 7B. For each trial-selection
and tuning reading, the leave-one-condition-out performance of every included unit is
computed in the peri-movement space (k = 1 to 14, window from -150 ms, and from 0 ms for
Figure 7B), the kinematic space (k = 1 to 14) and the eight maze task spaces. EMG was not
recorded for J-array, and `targets + instructed speed` does not exist for the maze task.

In [ ]:
t0 = time.time()
peri_rows, kin_rows, task_rows = [], [], []
for (proto, combine, onset), ctx in ctxs.items():
    for tun in ps.READINGS["tuning"]:
        inc = ps.tuning(ctx["prep"], tun) > ps.ARRAY_MIN_TUNING
        X = ctx["prep"][inc]
        base = {"proto": proto, "combine": combine, "onset": onset, "tuning": tun, "n_units": int(inc.sum())}
        for t_units in ps.READINGS["t_units"]:
            for fold in ps.READINGS["fold"]:
                for start in [-150, 0]:
                    err = ps.peri_errors(ctx, inc, t_units, fold, 14, start)
                    for k in range(1, 15):
                        pf = ps.perf_both(err[:, k - 1], X)
                        for norm in ps.READINGS["norm"]:
                            peri_rows.append({**base, "t_units": t_units, "fold": fold, "start": start,
                                              "k": k, "norm": norm, "perf": pf[norm]})
        for fold in ps.READINGS["fold"]:
            for kin_end in ps.READINGS["kin_end"]:
                err = ps.kinematic_errors(ctx, inc, fold, kin_end)
                for k in range(1, 15):
                    pf = ps.perf_both(err[:, k - 1], X)
                    for norm in ps.READINGS["norm"]:
                        kin_rows.append({**base, "fold": fold, "kin_end": kin_end, "k": k, "norm": norm, "perf": pf[norm]})
        for task_end in ps.READINGS["task_end"]:
            for split in ps.READINGS["split"]:
                errs, dims = ps.task_errors(ctx, inc, task_end, split)
                for name, err in errs.items():
                    pf = ps.perf_both(err, X)
                    for norm in ps.READINGS["norm"]:
                        task_rows.append({**base, "task_end": task_end, "split": split, "space": name,
                                          "k": dims[name], "norm": norm, "perf": pf[norm]})
peri_df, kin_df, task_df = pd.DataFrame(peri_rows), pd.DataFrame(kin_rows), pd.DataFrame(task_rows)
print(f"Grid computed in {time.time() - t0:.0f} s: {len(peri_df)} peri-movement, {len(kin_df)} kinematic, "
      f"{len(task_df)} task-space values")

### fig6-peri-movement-space-best

> For every dataset, preferred directions capture preparatory tuning best (highest
> cross-validated performance) when expressed in the peri-movement space, usually at middle
> dimensionalities.

Reproduced when the peri-movement space's best mean performance over k is higher than that of
every other space under one combination of the readings.

In [ ]:
PREP_KEYS = ["proto", "combine", "onset", "tuning"]

def combos():
    names = ["proto", "combine", "onset", "tuning", "t_units", "fold", "norm", "kin_end", "task_end", "split", "kmax"]
    for vals in itertools.product(*[ps.READINGS[n] for n in names]):
        yield dict(zip(names, vals))

pi = peri_df[peri_df.start == -150].set_index(PREP_KEYS + ["t_units", "fold", "norm", "k"]).perf.sort_index()
pi0 = peri_df[peri_df.start == 0].set_index(PREP_KEYS + ["t_units", "fold", "norm", "k"]).perf.sort_index()
ki = kin_df.set_index(PREP_KEYS + ["fold", "kin_end", "norm", "k"]).perf.sort_index()
ti = task_df.set_index(PREP_KEYS + ["task_end", "split", "norm", "space"]).sort_index()

def curves_for(c):
    pk = tuple(c[k] for k in PREP_KEYS)
    ks = np.arange(1, c["kmax"] + 1)
    peri = pi.loc[pk + (c["t_units"], c["fold"], c["norm"])].loc[ks]
    peri0 = pi0.loc[pk + (c["t_units"], c["fold"], c["norm"])].loc[ks]
    kin = ki.loc[pk + (c["fold"], c["kin_end"], c["norm"])].loc[ks]
    task = ti.loc[pk + (c["task_end"], c["split"], c["norm"])]
    return peri, peri0, kin, task

rows = []
for c in combos():
    peri, peri0, kin, task = curves_for(c)
    best_other = max(kin.max(), task.perf.max())
    rows.append({**c, "peri_best": peri.max(), "peri_best_k": int(peri.idxmax()), "peri_k1": peri.loc[1],
                 "peri_kmax": peri.loc[c["kmax"]],
                 "kin_best": kin.max(), "kin_best_k": int(kin.idxmax()), "kin_k1": kin.loc[1], "kin_kmax": kin.loc[c["kmax"]],
                 "task_best": task.perf.max(), "task_best_space": task.perf.idxmax(),
                 "n_task_above_0": int((task.perf > 0).sum()),
                 "peri0_best": peri0.max(), "peri0_best_k": int(peri0.idxmax()),
                 "peri_beats_all": peri.max() > best_other, "margin": peri.max() - best_other})
grid = pd.DataFrame(rows)
MAIN = ps.MAIN
is_main = (grid[list(MAIN)] == pd.Series(MAIN)).all(1)
print("Combinations of readings:", len(grid), "; peri-movement space best in", int(grid.peri_beats_all.sum()))
print("\nMain reading:")
print(grid[is_main].T.to_string())
print("\nShare of combinations where the peri-movement space is best, by each reading:")
for n in ["proto", "combine", "onset", "tuning", "t_units", "fold", "norm", "kin_end", "task_end", "split", "kmax"]:
    print(f"  {n}: " + ", ".join(f"{v} {grid[grid[n] == v].peri_beats_all.mean():.2f}" for v in ps.READINGS[n]))
print("\nMargin (peri best minus best other) over combinations: min", round(grid.margin.min(), 3),
      "median", round(grid.margin.median(), 3), "max", round(grid.margin.max(), 3))
print("Peri-movement best k over combinations:", grid.peri_best_k.value_counts().sort_index().to_dict())

In [ ]:
c = dict(MAIN)
peri, peri0, kin, task = curves_for(c)
pk = tuple(c[k] for k in PREP_KEYS)
n_inc = int(peri_df.set_index(PREP_KEYS).loc[pk].n_units.iloc[0])
print(f"Main reading, {n_inc} included units. Mean performance by k:")
print(pd.DataFrame({"peri-movement": peri, "kinematic": kin}).round(3).T.to_string())
print("\nTask spaces (dimensionality, mean performance):")
print(task[["k", "perf"]].round(4).to_string())
alt = dict(MAIN, fold="refit")
p2, _, k2, t2 = curves_for(alt)
print("\nVariant fold = refit:")
print(pd.DataFrame({"peri-movement": p2, "kinematic": k2}).round(3).T.to_string())
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (pp, kk, tk, title) in zip(axs, [(peri, kin, task, "space computed once"), (p2, k2, t2, "space refit per fold")]):
    ax.plot(pp.index, pp.values, "k-o", ms=3, label="peri-movement")
    ax.plot(kk.index, kk.values, "b-o", ms=3, label="kinematic")
    ax.plot(tk.k, tk.perf, "gs", label="task spaces")
    ax.axhline(0, color="0.7"); ax.set_ylim(-0.5, 0.5); ax.set_xlabel("dimensionality"); ax.set_title(title)
axs[0].set_ylabel("mean performance"); axs[0].legend(fontsize=7)
plt.show()
allv = np.r_[peri.values, kin.values, task.perf.values, p2.values, k2.values, t2.perf.values]
print("Values below -0.5 (outside the axes):", int((allv < -0.5).sum()), "; min", round(allv.min(), 3))

**Result: reproduced.** Under the main reading the peri-movement space's best mean performance is
0.351 (k = 10), above the kinematic space (0.103, k = 4) and the best task space (kitchen sink,
0.250). Over the 2048 combinations of readings the peri-movement space is best in 1468; the
criterion asks for one. It is best in 95% of the combinations with tuning as a range and 49% with
the SD (6 or 7 units), and in 94% with all units in the T-matrix against 50% with included units
only. Its best k is 7 or more in most combinations (from 2 to 14), and 10 under the main reading.
EMG was not recorded for J-array.

### fig6-each-dimensionality

> At each dimensionality taken alone the peri-movement space almost always performs best; ...
> endpoints + maximum speed does not perform well for the maze task.

Every other space at each of its dimensionalities against the peri-movement space at the same
k; reproduced when the peri-movement space wins at least 80% and beats endpoints + maximum
speed at k = 3.

In [ ]:
rows = []
for c in combos():
    peri, peri0, kin, task = curves_for(c)
    wins = [peri.loc[k] > kin.loc[k] for k in kin.index]
    wins += [peri.loc[int(r.k)] > r.perf for _, r in task.iterrows() if r.k <= c["kmax"]]
    e3 = task.loc["endpoints + max speed"].perf
    rows.append({**c, "comparisons": len(wins), "won": int(sum(wins)), "share": np.mean(wins),
                 "peri_k3": peri.loc[3], "endpoints_max_speed": e3, "beats_e3": peri.loc[3] > e3})
each = pd.DataFrame(rows)
each["criterion_met"] = (each.share >= 0.8) & each.beats_e3
print("Main reading:", each[is_main.values][["comparisons", "won", "share", "peri_k3", "endpoints_max_speed", "criterion_met"]].round(3).to_dict("records"))
print("Combinations meeting the criterion:", int(each.criterion_met.sum()), "of", len(each))
print("Share won over combinations: min", round(each.share.min(), 3), "median", round(each.share.median(), 3), "max", round(each.share.max(), 3))
print("By reading, fraction of combinations meeting the criterion:")
for n in ["proto", "combine", "onset", "tuning", "t_units", "fold", "norm", "kin_end", "task_end", "split", "kmax"]:
    print(f"  {n}: " + ", ".join(f"{v} {each[each[n] == v].criterion_met.mean():.2f}" for v in ps.READINGS[n]))
peri, _, kin, task = curves_for(dict(MAIN))
print("\nMain reading, kinematic minus peri-movement by k:", (kin - peri).round(3).to_dict())
# the main reading with the space refit without the left-out condition
cr = dict(MAIN, fold="refit")
peri_r, _, kin_r, task_r = curves_for(cr)
print("\nMain reading with fold=refit, mean performance by k:")
print(pd.DataFrame({"peri-movement": peri_r, "kinematic": kin_r}).round(3).T.to_string())
print("Task spaces (refit does not apply):", task_r.perf.round(3).to_dict())
print("Main reading with fold=refit:", each[(each[list(cr)] == pd.Series(cr)).all(axis=1)][
    ["comparisons", "won", "share", "peri_k3", "endpoints_max_speed", "criterion_met"]].round(3).to_dict("records"))
print("Comparisons lost under fold=refit, main reading otherwise:",
      [f"kinematic k={k}" for k in kin_r.index if not peri_r.loc[k] > kin_r.loc[k]]
      + [f"{s} k={int(r.k)}" for s, r in task_r.iterrows() if r.k <= cr["kmax"] and not peri_r.loc[int(r.k)] > r.perf])

**Result: reproduced under the main reading (an assumption).** The peri-movement space wins 20 of
22 comparisons (0.909) and beats endpoints + max speed at k = 3 (0.126 against 0.0003). Over all
combinations the criterion is met in 1074 of 2048. It depends most on the trial selection (met in
69% with `concat` and 36% with `each`), the tuning reading (75% and 30%), the units of the
T-matrix (67% and 38%) and the fold reading (37% with the space computed once, 68% refit).
With the space refit and the main reading otherwise, the peri-movement space wins 21 of 22
comparisons (it loses only to the kinematic space at k = 1) and beats endpoints + max speed at
k = 3 (0.132 against 0.0003), so the criterion is also met there.

### fig6-dimensionality-trend

> Performance of the PCA-based spaces typically rises with dimensionality and falls beyond
> about 10 dimensions, a sign of overfitting.

Reproduced when, for the peri-movement space, performance at its best k exceeds that at k = 1,
the best k is 10 or less, and performance at the largest tested k is below the best.

In [ ]:
trend = grid.copy()
trend["peri_rises"] = trend.peri_best > trend.peri_k1
trend["peri_best_k_le_10"] = trend.peri_best_k <= 10
trend["peri_falls"] = trend.peri_kmax < trend.peri_best
trend["criterion_met"] = trend.peri_rises & trend.peri_best_k_le_10 & trend.peri_falls
trend["kin_met"] = (trend.kin_best > trend.kin_k1) & (trend.kin_best_k <= 10) & (trend.kin_kmax < trend.kin_best)
print("Main reading:", trend[is_main][["peri_k1", "peri_best", "peri_best_k", "peri_kmax", "criterion_met",
                                        "kin_k1", "kin_best", "kin_best_k", "kin_kmax", "kin_met"]].round(3).to_dict("records"))
print("Combinations meeting the criterion (peri-movement):", int(trend.criterion_met.sum()), "of", len(trend),
      "; kinematic:", int(trend.kin_met.sum()))
print("By reading, fraction meeting the criterion:")
for n in ["proto", "combine", "onset", "tuning", "t_units", "fold", "norm", "kmax"]:
    print(f"  {n}: " + ", ".join(f"{v} {trend[trend[n] == v].criterion_met.mean():.2f}" for v in ps.READINGS[n]))
print("\nBest k of the peri-movement space, by fold reading:")
print(trend.groupby("fold").peri_best_k.describe().to_string())

**Result: reproduced under the main reading (an assumption).** Peri-movement performance rises
from 0.056 at k = 1 to 0.351 at k = 10 and falls to -0.456 at k = 14; the kinematic space does
the same (0.061 at k = 1, best 0.103 at k = 4, -0.451 at k = 14). The fall depends on computing
the space once: the criterion is met in 91% of the combinations with the space computed once and
6% when it is refit without the left-out condition. Refit, with the main reading otherwise,
peri-movement performance rises at every k from 0.054 at k = 1 to 0.461 at k = 14 (printed under
fig6-each-dimensionality), and the best k is 10 to 14 over those combinations. With the space
computed once, 14 components are fit from 27 conditions, so the fall may reflect the small
number of conditions here (the paper had 108); the outcome rests on the fold reading.

### fig6-pd-adequate-all-spaces

> The PD performs adequately (above chance, performance above zero) in most spaces.

Reproduced when mean performance is above zero for the peri-movement and kinematic spaces at
their best k and for more than half of the task spaces.

In [ ]:
ad = grid.copy()
ad["criterion_met"] = (ad.peri_best > 0) & (ad.kin_best > 0) & (ad.n_task_above_0 > 4)
print("Main reading:", ad[is_main][["peri_best", "kin_best", "n_task_above_0", "criterion_met"]].round(3).to_dict("records"))
print("Combinations meeting the criterion:", int(ad.criterion_met.sum()), "of", len(ad))
print("Task spaces above zero over combinations:", ad.n_task_above_0.value_counts().sort_index().to_dict(), "(of 8)")
print("By reading, fraction meeting the criterion:")
for n in ["proto", "combine", "onset", "tuning", "fold", "norm", "kin_end", "task_end", "split"]:
    print(f"  {n}: " + ", ".join(f"{v} {ad[ad[n] == v].criterion_met.mean():.2f}" for v in ps.READINGS[n]))
print("\nPer task space, share of combinations with mean performance above zero:")
print(task_df.groupby("space").perf.apply(lambda x: (x > 0).mean()).round(2).to_string())
print("\nPer task space, main reading: performance per unit above zero (count of units):")
ctx = ctxs[("a", "concat", "a")]
inc = ps.tuning(ctx["prep"], "a") > 5
errs, dims = ps.task_errors(ctx, inc, "speed", "length")
print({k: f"{int((ps.performance(e, ctx['prep'][inc]) > 0).sum())} of {int(inc.sum())}" for k, e in errs.items()})

**Result: reproduced under the main reading (an assumption).** Mean performance is above zero for
the peri-movement space (0.351) and the kinematic space (0.103) at their best k, and for 7 of the
8 task spaces; endpoints + max velocity is negative (-0.125) and endpoints + max speed is just
above zero (0.0003). The criterion is met in 1936 of 2048 combinations.

## fig6e-summary-rank

Not tested: the summary combines monkeys A, B and J (single-electrode), whose data and EMG
are not in the dandiset.

## fig7a-shuffled-space-poor

> A control peri-movement space in which each neuron's peri-movement pattern is inverted on a
> random half of conditions performs poorly.

For each unit and condition, with probability 0.5 the movement-locked response after -150 ms
is reflected about the rate at -150 ms; the PCA space is then built as for the peri-movement
space. 20 draws (a standard choice), averaged. Run under the main reading and, one at a time,
under the other value of each reading that changes the peri-movement space.

In [ ]:
def shuffled_move(move, rng):
    i0 = ps.T_WINDOW[0] - ps.MOVE_TRACE[0]
    flip = rng.random(move.shape[:2]) < 0.5
    out = move.copy()
    ref = move[:, :, i0:i0 + 1]
    out[:, :, i0:] = np.where(flip[:, :, None], 2 * ref - move[:, :, i0:], move[:, :, i0:])
    return out

variants = [dict(MAIN)] + [dict(MAIN, **{n: ps.READINGS[n][1]}) for n in ["combine", "proto", "onset", "t_units", "fold", "norm", "kin_end"]]
rng7a = np.random.default_rng(7)
rows = []
t0 = time.time()
for v in variants:
    ctx = ctxs[(v["proto"], v["combine"], v["onset"])]
    inc = ps.tuning(ctx["prep"], v["tuning"]) > ps.ARRAY_MIN_TUNING
    X = ctx["prep"][inc]
    n_draws = 20
    shuf = np.zeros(14)
    for d in range(n_draws):
        err = ps.peri_errors(ctx, inc, v["t_units"], v["fold"], 14, move=shuffled_move(ctx["move"], rng7a))
        shuf += np.array([ps.perf_both(err[:, k], X)[v["norm"]] for k in range(14)]) / n_draws
    peri, _, kin, task = curves_for(v)
    rows.append({"variant": ", ".join(f"{n}={v[n]}" for n in v if v[n] != MAIN[n]) or "main", "draws": n_draws,
                 "shuffled_best": shuf.max(), "shuffled_best_k": int(shuf.argmax() + 1),
                 "peri_best": peri.max(), "kin_best": kin.max(),
                 "below_peri_every_k": bool(np.all(shuf < peri.values[:14])),
                 "below_kin_best": shuf.max() < kin.max(),
                 "shuffled_by_k": np.round(shuf, 3).tolist()})
fig7a = pd.DataFrame(rows)
fig7a["criterion_met"] = fig7a.below_peri_every_k & fig7a.below_kin_best
print(f"computed in {time.time() - t0:.0f} s")
print(fig7a.drop(columns="shuffled_by_k").round(3).to_string(index=False))
print("\nShuffled-space performance by k:")
for _, r in fig7a.iterrows():
    print(f"  {r.variant}: {r.shuffled_by_k}")

**Result: reproduced under the main reading (an assumption).** The shuffled space's best mean
performance is 0.014 (k = 3), below the peri-movement space at every k and below the kinematic
space's best (0.103). With 20 draws in every variant, the criterion holds in 7 of 8 variants. It
fails with the `each` selection, where the shuffled space is not below the peri-movement space at
every k. With the space refit per fold it holds by a narrow margin: the shuffled best is 0.159,
against a kinematic best of 0.160 and a peri-movement best of 0.461.

## fig7b-window-start-onset

> Sliding the start of the peri-movement window from 150 ms before movement onset to movement
> onset barely lowers performance, which stays at 97% (monkey J) and 94% (monkeys A, B) of its
> original value.

Mean performance of the peri-movement space with the window from 0 to +800 ms, divided by
that from -150 ms, each at its best k. Only reading (a) of the claims file (J-array against
0.97) can be run; the J single-electrode dataset and monkeys A and B are not in the dandiset.

In [ ]:
fb = grid.copy()
fb["ratio"] = fb.peri0_best / fb.peri_best
fb["within_tol"] = (fb.ratio - 0.97).abs() <= 0.097
print("Expected 0.97 +/- 0.097 (0.873 to 1.067)")
print("Main reading:", fb[is_main][["peri_best", "peri_best_k", "peri0_best", "peri0_best_k", "ratio", "within_tol"]].round(3).to_dict("records"))
print("Combinations within tolerance:", int(fb.within_tol.sum()), "of", len(fb))
print("Ratio over combinations: min", round(fb.ratio.min(), 3), "median", round(fb.ratio.median(), 3), "max", round(fb.ratio.max(), 3))
print("By reading, median ratio and fraction within tolerance:")
for n in ["proto", "combine", "onset", "tuning", "t_units", "fold", "norm", "kmax"]:
    print(f"  {n}: " + ", ".join(f"{v} {fb[fb[n] == v].ratio.median():.3f}/{fb[fb[n] == v].within_tol.mean():.2f}" for v in ps.READINGS[n]))
# finer sweep of the start under the main reading
ctx = ctxs[("a", "concat", "a")]
inc = ps.tuning(ctx["prep"], "a") > 5
X = ctx["prep"][inc]
sweep = {}
for start in [-150, -100, -50, 0, 50, 100]:
    err = ps.peri_errors(ctx, inc, "all", "once", 14, start)
    pf = [ps.perf_both(err[:, k], X)["neuron"] for k in range(14)]
    sweep[start] = (round(max(pf), 3), int(np.argmax(pf) + 1))
print("\nMain reading, best performance (and k) by window start (ms from onset):", sweep)

**Result: reproduced under the main reading (an assumption), for J-array only.** With the window
starting at movement onset, the best performance is 0.318 (k = 9), against 0.351 (k = 10) from
-150 ms, a ratio of 0.908, inside 0.873 to 1.067. Over all combinations the ratio is within
tolerance in 1336 of 2048 (median 0.917; within tolerance in 47% with the space computed once and
84% refit). The 0.94 for monkeys A and B cannot be tested.

## fig7c-anterior-posterior

> The peri-movement space captures preparatory tuning at both anterior and posterior sites,
> with normalized performance only slightly different from that of random halves of the data,
> slightly better at anterior sites.

Anterior and posterior are the two arrays, from the unit id prefix (prefix 1 assumed PMd,
anterior), as in results-ap-location-no-effect: with array membership as the AP location, the
units at or above the median are one array and those at or below it the other. For each
subset, the peri-movement space is built from that subset's units only (the unit being fit
excluded) and performance computed at k = 6, normalized by the upper limit, the PCA space of
the subset's preparatory rates, at k = 6 or at its best k. The upper-limit space leaves out
the unit being fit: the Supplemental Experimental Procedures justify it with "whatever one neuron
is tuned for, it ought to be shared with at least some of the other neurons", which compares a
neuron with the others. With the unit's own rate in the space, the fit is to a space that
contains the rate itself; these rows are shown but do not count toward the outcome. Random
halves: the units split at random into two halves, 20 draws (a standard choice), 40 halves; their
spread is printed so that the 10% bound can be compared with sampling noise.

In [ ]:
def subset_scores(ctx, members, inc, fold, t_units, self_in_upper):
    """Mean performance at k = 6 in the peri-movement space of `members`, and of the upper-limit
    space (k = 1..14), over included units of the subset."""
    fit = members & inc
    in_t = members if t_units == "all" else fit
    X = ctx["prep"][fit]
    Gp = ps.neuron_grams(ps.t_matrix(ctx["move"]))
    G = Gp[in_t].sum(0)
    err = ps.loo_pca_space(G, Gp[fit] * in_t[fit][:, None, None], X, 6, fold)
    peri6 = np.mean(ps.performance(err[:, 5], X))
    P = ctx["prep"][:, :, None]
    Pp = ps.neuron_grams(P)
    Gu = Pp[members].sum(0)
    parts = None if self_in_upper else Pp[fit]
    if self_in_upper:
        erru = ps.loo_pca_space(Gu, None, X, 14, fold)
    else:
        erru = ps.loo_pca_space(Gu, parts, X, 14, fold)
    upper = np.array([np.mean(ps.performance(erru[:, k], X)) for k in range(14)])
    return peri6, upper, int(fit.sum())

rng7c = np.random.default_rng(17)
rows = []
for v in [dict(MAIN), dict(MAIN, fold="refit"), dict(MAIN, combine="each"), dict(MAIN, t_units="included")]:
    ctx = ctxs[(v["proto"], v["combine"], v["onset"])]
    inc = ps.tuning(ctx["prep"], v["tuning"]) > ps.ARRAY_MIN_TUNING
    for self_in in [False, True]:
        res = {}
        for name, members in [("anterior", units.id_array.values == 1), ("posterior", units.id_array.values == 2),
                              ("all", np.ones(len(units), bool))]:
            res[name] = subset_scores(ctx, members, inc, v["fold"], v["t_units"], self_in)
        rand = []
        for d in range(20):
            perm = rng7c.permutation(len(units))
            for half in (perm[: len(units) // 2], perm[len(units) // 2:]):
                m = np.zeros(len(units), bool); m[half] = True
                rand.append(subset_scores(ctx, m, inc, v["fold"], v["t_units"], self_in))
        for up in ["k6", "best"]:
            def norm(p, u):
                return p / (u[5] if up == "k6" else u.max())
            r_all = np.array([norm(p, u) for p, u, _ in rand])
            r_norm = r_all.mean()
            a_norm, p_norm = norm(*res["anterior"][:2]), norm(*res["posterior"][:2])
            rows.append({"variant": ", ".join(f"{n}={v[n]}" for n in v if v[n] != MAIN[n]) or "main",
                         "self_in_upper": self_in, "upper": up,
                         "n_anterior": res["anterior"][2], "n_posterior": res["posterior"][2],
                         "all_norm": norm(*res["all"][:2]), "anterior_norm": a_norm, "posterior_norm": p_norm,
                         "random_half_norm": r_norm, "random_half_sd": r_all.std(ddof=1),
                         "random_half_p5": np.percentile(r_all, 5), "random_half_p95": np.percentile(r_all, 95),
                         "anterior_raw": res["anterior"][0], "posterior_raw": res["posterior"][0],
                         "random_raw": np.mean([p for p, _, _ in rand]),
                         "ant_vs_random": a_norm / r_norm, "post_vs_random": p_norm / r_norm})
fig7c = pd.DataFrame(rows)
fig7c["criterion_met"] = ((fig7c.anterior_norm > 0) & (fig7c.posterior_norm > 0)
                          & ((fig7c.ant_vs_random - 1).abs() <= 0.1) & ((fig7c.post_vs_random - 1).abs() <= 0.1))
fig7c["anterior_better"] = fig7c.anterior_norm > fig7c.posterior_norm
print(fig7c.round(3).to_string(index=False))
print("Rows meeting the criterion, unit being fit left out of the upper-limit space (counted):",
      int(fig7c[~fig7c.self_in_upper].criterion_met.sum()), "of", int((~fig7c.self_in_upper).sum()))
print("Rows meeting the criterion, unit included (shown, not counted):",
      int(fig7c[fig7c.self_in_upper].criterion_met.sum()), "of", int(fig7c.self_in_upper.sum()))
print("Anterior and posterior normalized performance against the 5th to 95th percentile of the 40 random halves:")
print(fig7c[["variant", "self_in_upper", "upper", "anterior_norm", "posterior_norm", "random_half_p5", "random_half_p95"]]
      .assign(anterior_inside=lambda d: d.anterior_norm.between(d.random_half_p5, d.random_half_p95),
              posterior_inside=lambda d: d.posterior_norm.between(d.random_half_p5, d.random_half_p95))
      .round(3).to_string(index=False))

**Result: not reproduced under the main reading (an assumption).** With the unit being fit left
out of the upper-limit space, at k = 6, normalized performance is 0.785 for the anterior array
(prefix 1) and 0.814 for the posterior array, against 0.961 for random halves: ratios of 0.817
and 0.847, both outside the 10% bound. None of the 8 counted combinations meets the criterion;
the 2 that do both put the unit's own rate in its upper-limit space, which is shown but not
counted. Under the main reading anterior is below posterior (0.785 against 0.814), where the
paper reports anterior slightly better. The 40 random halves spread widely (SD 0.190, 5th to 95th
percentile 0.746 to 1.252), and both arrays lie inside that spread at k = 6, so the 10% bound is
narrower than the sampling noise of a random half here. Under `each` at k = 6 the random-half
mean is -0.826 with SD 10.8, as an upper limit near zero in some half makes the ratio unstable.
The split rests on the assumed array of each unit.

## fig7d-pd-consistency

> In all four datasets and for every comparison, the PD is more stable across random subsets
> of conditions when measured in the peri-movement space than in task, kinematic or EMG spaces.

Per unit, 25 conditions drawn with replacement, the unit-length PD fitted, another 25 drawn
and the PD refitted; consistency is the dot product, averaged over 100 draws (standard choice)
and units. Compared: endpoints + maximum speed (3-D) and segment directions (6-D) against the
peri-movement space at 3 and 6 dimensions, and the kinematic space at its best k against the
peri-movement space at that k. The spaces are taken as computed from all conditions. With 27 conditions, a draw of 25 with
replacement holds about 16 distinct conditions and two draws share about 10 (printed below),
so the two PDs are fitted to overlapping data more than with the paper's 108 conditions. Whether
the axes of a space are scaled before the PD is fitted is not stated; the spaces as they are
(PC scores; task variables in mm and mm/s) and with each axis scaled to unit SD are both run.

In [ ]:
def consistency(S, X, rng, n_draws=100):
    c = S.shape[0]
    D = np.column_stack([np.ones(c), S])
    vals = np.zeros(X.shape[0])
    for _ in range(n_draws):
        i1, i2 = rng.integers(0, c, 25), rng.integers(0, c, 25)
        b1 = np.linalg.lstsq(D[i1], X[:, i1].T, rcond=None)[0][1:]
        b2 = np.linalg.lstsq(D[i2], X[:, i2].T, rcond=None)[0][1:]
        b1 /= np.linalg.norm(b1, axis=0, keepdims=True)
        b2 /= np.linalg.norm(b2, axis=0, keepdims=True)
        vals += (b1 * b2).sum(0) / n_draws
    return vals

rng7d = np.random.default_rng(23)
rows = []
for v in [dict(MAIN), dict(MAIN, combine="each"), dict(MAIN, t_units="included"), dict(MAIN, task_end="fixed"),
          dict(MAIN, split="time"), dict(MAIN, kin_end="speed"), dict(MAIN, onset="b"), dict(MAIN, proto="b")]:
    ctx = ctxs[(v["proto"], v["combine"], v["onset"])]
    inc = ps.tuning(ctx["prep"], v["tuning"]) > ps.ARRAY_MIN_TUNING
    X = ctx["prep"][inc]
    spaces, _ = ps.task_spaces(ctx["pos"], ctx["vel"], ctx["times"], trials, v["task_end"], v["split"])
    _, _, kin, _ = curves_for(v)
    kbest = int(kin.idxmax())
    KT = ps.kinematic_matrix(ctx["pos"], ctx["vel"], ctx["times"], ps.kinematic_end_ms(ctx, v["kin_end"]))
    Skin = ps._scores_from_gram(ps.centre_gram(KT @ KT.T), 14)[0][:, :kbest]
    Gp = ps.neuron_grams(ps.t_matrix(ctx["move"]))
    in_t = np.ones(len(inc), bool) if v["t_units"] == "all" else inc
    G = Gp[in_t].sum(0)
    for scale in ["raw", "unit SD"]:
        def sc(S):
            return S / S.std(0) if scale == "unit SD" else S
        comps = [("endpoints + max speed", sc(spaces["endpoints + max speed"]), 3),
                 ("segment directions", sc(spaces["segment directions"]), 6),
                 (f"kinematic k={kbest}", sc(Skin), kbest)]
        for name, S, k in comps:
            other = consistency(S, X, rng7d)
            peri_c = np.zeros(X.shape[0])
            for j, u in enumerate(np.nonzero(inc)[0]):
                Gj = G - Gp[u] * in_t[u]
                Sp = sc(ps._scores_from_gram(ps.centre_gram(Gj), k)[0])
                peri_c[j] = consistency(Sp, X[j:j + 1], rng7d)[0]
            rows.append({"variant": ", ".join(f"{n}={v[n]}" for n in v if v[n] != MAIN[n]) or "main",
                         "scale": scale, "space": name, "k": k, "other_consistency": other.mean(),
                         "peri_consistency": peri_c.mean(), "peri_higher": peri_c.mean() > other.mean(),
                         "units_peri_higher": int((peri_c > other).sum()), "n_units": len(other)})
fig7d = pd.DataFrame(rows)
# how much two draws of 25 of the 27 conditions with replacement overlap
ov = np.random.default_rng(24)
d1 = [set(ov.integers(0, 27, 25)) for _ in range(10000)]
d2 = [set(ov.integers(0, 27, 25)) for _ in range(10000)]
print(f"Distinct conditions per draw: mean {np.mean([len(a) for a in d1]):.1f} of 27; "
      f"shared by two draws: mean {np.mean([len(a & b) for a, b in zip(d1, d2)]):.1f}")
print(fig7d.round(3).to_string(index=False))
by = fig7d.groupby(["variant", "scale"]).peri_higher.all()
print("\nPeri-movement higher in every comparison, by variant and scaling:")
print(by.to_string())

**Result: reproduced, for J-array.** Under the main reading the peri-movement space has the higher
mean consistency in all three comparisons: 0.756 against 0.610 (endpoints + max speed, k = 3),
0.684 against 0.570 (segment directions, k = 6) and 0.748 against 0.666 (kinematic, k = 4). It is
higher in every comparison under all eight variants and with the axes as they are or scaled to
unit SD. Two draws share about 10 of their 16 distinct conditions, so consistency here is higher
than it would be with 108 conditions in both kinds of space. The other three datasets, and EMG, are not in the dandiset.

## fig8-oscillator-model

> In a simulated two-unit linear oscillator held at 25 initial states and then released,
> preparatory and peri-movement activity of a unit are nearly uncorrelated (r = 0.08), yet a
> PD in the two-dimensional peri-movement space captures the preparatory tuning perfectly.

A simulation; no recorded data. W = [1 -2pi/360; 2pi/360 1], 25 initial states held, then
released. Readings: initial states uniform in [-1, 1]^2 or standard Gaussian; gray interval
one cycle (360 steps) or half (180). 1000 draws each. The PD part: leave-one-out performance
of a PD for unit 1's initial rate in the 2-D PCA space of the released activity of both units.

In [ ]:
rng8 = np.random.default_rng(8)
rows = []
for init in ["uniform", "gaussian"]:
    for release in [360, 180]:
        rs, perfs = [], []
        for d in range(1000):
            act, x0 = ps.oscillator(rng8, 25, 50, release, init)
            rel = act[:, 50:, :]
            rs.append(np.corrcoef(x0[:, 0], rel[:, :, 0].mean(1))[0, 1])
            T = rel.transpose(0, 2, 1).reshape(25, -1)
            S = ps._scores_from_gram(ps.centre_gram(T @ T.T), 2)[0]
            err = ps.loo_errors_fixed(S, x0[None, :, 0])
            perfs.append(ps.performance(err, x0[None, :, 0])[0])
        rs, perfs = np.array(rs), np.array(perfs)
        lo, hi = np.percentile(rs, [2.5, 97.5])
        rows.append({"init": init, "interval_steps": release, "median_r": np.median(rs), "r_2.5%": lo, "r_97.5%": hi,
                     "0.08_inside": lo <= 0.08 <= hi, "min_pd_perf": perfs.min(), "median_pd_perf": np.median(perfs),
                     "pd_ge_0.99": perfs.min() >= 0.99})
fig8 = pd.DataFrame(rows)
print(fig8.round(4).to_string(index=False))

**Result: reproduced.** Under every reading the median correlation of unit 1's held and released
activity is near zero (-0.019 to 0.012), with a central 95% from about -0.41 to 0.40, which
contains 0.08. A PD in the 2-D PCA space of the released activity captures unit 1's initial rate
with leave-one-out performance of 1.0 in every draw.

## Summary

| Claim | Outcome | Main numbers |
|---|---|---|
| fig4abc-tuning-correlation-single-electrode | blocked (data missing) | single-electrode data not in the dandiset |
| fig4d-tuning-correlation-j-array | partially reproduced (assumption) | mean r 0.195, Spearman 0.212, p = 0.023 |
| fig4e-correlation-time-course | not reproduced | r 0.218 at movement onset |
| results-ap-location-no-effect | reproduced (assumption) | r = 0.047, p = 0.844 (J-array) |
| supp-preparatory-dimensionality | reproduced (assumption) | 11 components |
| fig6-peri-movement-space-best | reproduced | 0.351 against 0.250 and 0.103 |
| fig6-each-dimensionality | reproduced (assumption) | 20 of 22 comparisons |
| fig6-dimensionality-trend | reproduced (assumption) | best k 10, falls to -0.456 at 14 |
| fig6-pd-adequate-all-spaces | reproduced (assumption) | 7 of 8 task spaces above 0 |
| fig6e-summary-rank | blocked (data missing) | monkeys A, B, J and EMG not in the dandiset |
| fig7a-shuffled-space-poor | reproduced (assumption) | shuffled best 0.014 |
| fig7b-window-start-onset | reproduced (assumption) | ratio 0.908 |
| fig7c-anterior-posterior | not reproduced (assumption) | anterior 0.817, posterior 0.847 of random halves |
| fig7d-pd-consistency | reproduced | higher in 3 of 3 comparisons |
| fig8-oscillator-model | reproduced | 0.08 inside the central 95% |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Only the array dataset of monkey J can be tested,
on 27 of its 108 conditions and 500 trials of one session, so every J-array number rests on a
quarter of the conditions and fewer trials per condition after selection (11 to 18). The paper
leaves open the proto-trial, how the velocity correlation combines the two axes, the movement
onset method, what "tuning" in spikes/s means, whether the PCA space is recomputed without the
left-out condition, which units form the T-matrix, the end of the kinematic window and of the
reach, how paths are split, and, for Figure 7, the upper-limit space and the scaling of axes. The upper-limit space of Figure
7C leaves out the unit being fit, as the paper's justification of it compares a neuron with the
others.
Each is run, and outcomes that differ between readings are marked as assumptions. The main
reading takes the mean proto-trial, the concatenated velocity profile, the trial table's movement
onset, tuning as the range of condition means, all units in the T-matrix, the space computed once
from all conditions (the paper describes leaving out a condition only for the PD fit), and
performance averaged over units. Figure 4E uses the movement-locked average, the part of the
paper's spliced trace that is locked to movement onset. The anterior and posterior split uses the
array of each unit, inferred from the unit id prefix.

**What was found about the dataset.**

- The units' electrode references all point to the PMd group: the referenced electrode is the
  channel in the unit id minus one, for both id prefixes, so the 79 units of prefix 2 reference
  PMd electrodes. Which prefix is which array is not stated; electrode coordinates are NaN.
- The dandiset holds a 500-trial, 27-condition subset of one session; the paper's J-array dataset
  had 108 conditions and 146 units (the file has 162), and there is no way to tell from the files
  whether the session is the paper's. The single-electrode datasets and EMG are not in it.
- Hand data are at 1 kHz within trials only, where the paper tracked the hand at 60 Hz; the
  resampling is not described. Velocity agrees with the derivative of position.
- The test file holds 100 unlabelled 700 ms segments of the held-in units, not usable for these
  claims; no test segment repeats a training interval sequence.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")